# NumPy Tutorial (all chapters)

This file is generated from the individual notebooks by `tools/build_complete.py`.

- 0. Data Types
- 1. Create an Array
- 2. Inspect an Array
- 3. Sampling Methods
- 4. Math Functions
- 5. Slicing & Indexing
- 6. Sort an Array
- 7. Manipulate an Array
- 8. Combine & Split an Array
- 9. Set Operations
- 10. Linear Algebra
- 11. Broadcasting
- 12. Views and Copies
- 13. Vectorization & Ufuncs
- 14. Missing Values
- 15. Saving & Loading Arrays


# 0. Data Types

Every NumPy array holds elements of a single data type (its `dtype`). Picking the right dtype matters for memory use, speed and correctness, so it's worth getting comfortable with them before anything else.

In [1]:
import numpy as np
np.__version__

'2.5.3'

## The main families

| Kind | Types | Type codes |
|---|---|---|
| Signed integers | `int8`, `int16`, `int32`, `int64` | `i1`, `i2`, `i4`, `i8` |
| Unsigned integers | `uint8`, `uint16`, `uint32`, `uint64` | `u1`, `u2`, `u4`, `u8` |
| Floating point | `float16`, `float32`, `float64` | `f2`, `f4`, `f8` |
| Complex | `complex64`, `complex128` | `c8`, `c16` |
| Boolean | `bool` | `?` |
| Fixed-length bytes | `bytes_` | `S` (e.g. `S5`) |
| Fixed-length unicode | `str_` | `U` (e.g. `U10`) |
| Python objects | `object_` | `O` |

The number in the type name is the size in **bits**; the number in the type code is the size in **bytes**. So `float32` and `f4` are the same thing.

There's also `float128` / `longdouble`, but its actual precision depends on your platform (on Windows it's just `float64`), so don't rely on it for portable code.

## Choosing a dtype

You can pass the dtype as a name string, a type code string, or a NumPy type object. They all mean the same thing:

In [2]:
a = np.array([1, 2, 3], dtype='float32')
b = np.array([1, 2, 3], dtype='f4')
c = np.array([1, 2, 3], dtype=np.float32)
a.dtype, b.dtype, c.dtype

(dtype('float32'), dtype('float32'), dtype('float32'))

In [3]:
z = np.array([1+2j, 3-4j], dtype='c8')
z.dtype

dtype('complex64')

Type codes only work as **strings**. There's no `np.c8` or `np.f4` attribute:

In [4]:
try:
    np.array([1+2j, 3-4j], dtype=np.c8)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: module 'numpy' has no attribute 'c8'


If you don't specify a dtype, NumPy picks one from the data. Integers default to `int64` on most platforms and floats to `float64`:

In [5]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)
print(np.array([True, False]).dtype)
print(np.array(['a', 'bc']).dtype)

int64
float64
bool
<U2


## Type limits

`np.iinfo` and `np.finfo` tell you the range and precision of a type. Handy when you're deciding whether a smaller type is safe to use.

In [6]:
print(np.iinfo(np.int8))
print(np.iinfo(np.uint16).max)
print(np.finfo(np.float32).eps)   # smallest step away from 1.0

Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

65535
1.1920929e-07


## Converting between types

`astype` gives you a **new** array with the requested dtype. The original isn't touched, so remember to assign the result.

In [7]:
arr = np.array([1, 2, 3], dtype='int16')
print('before:', arr.dtype)

arr = arr.astype(np.float32)
print('after: ', arr.dtype)

before: int16
after:  float32


Converting floats to integers truncates toward zero, it doesn't round:

In [8]:
np.array([1.7, -1.7, 2.5]).astype(int)

array([ 1, -1,  2])

### Watch out for overflow

Going down to a smaller type (say `int16` to `int8`) silently wraps values that don't fit. These bugs are really annoying to track down, so check ranges first.

In [9]:
arr = np.array([126, 127, 256], dtype='int16')
print('int16:', arr)

# int8 can only hold -128..127, so 256 wraps around to 0
print('int8: ', arr.astype('int8'))

int16: [126 127 256]
int8:  [126 127   0]


Arithmetic can overflow too. Here 200 + 100 doesn't fit in a `uint8`:

In [10]:
pixels = np.array([200, 250], dtype=np.uint8)
print(pixels + np.uint8(100))           # wraps around
print(pixels.astype(np.int16) + 100)    # upcast first if you need the real answer

[44 94]
[300 350]


## Type promotion

When you combine arrays of different types, NumPy promotes to a type that can hold both. Since NumPy 2.0, plain Python scalars are "weak": they adapt to the array's type instead of forcing an upcast.

In [11]:
i = np.array([1, 2, 3], dtype=np.int32)
f = np.array([0.5, 0.5, 0.5], dtype=np.float32)

print((i + f).dtype)        # int32 + float32 -> float64
print((f + 1.0).dtype)      # Python float doesn't upcast float32
print(np.result_type(np.int8, np.uint8))

float64
float32
int16


## Strings

String arrays are **fixed width**. `S3` means bytes of length 3, `U10` means unicode of length 10. If you don't give a length, NumPy uses the longest string in the data. Anything longer than the declared width gets cut off without warning.

In [12]:
print(np.array(['abc', 'defg'], dtype='S3'))
print(np.array(['abcd', 'efghi'], dtype='U3'))

[b'abc' b'def']
['abc' 'efg']


In [13]:
arr = np.array(['a', 'abc', 'abcd'])
print(arr, arr.dtype)    # <U4 because 'abcd' is the longest

['a' 'abc' 'abcd'] <U4


This also means assigning a longer string into an existing array truncates it:

In [14]:
arr[0] = 'something long'
arr

array(['some', 'abc', 'abcd'], dtype='<U4')

Since NumPy 2.0 there's also `np.dtypes.StringDType()`, a variable-length string type that avoids the truncation problem entirely:

In [15]:
arr = np.array(['a', 'abc'], dtype=np.dtypes.StringDType())
arr[0] = 'something long'
arr

array(['something long', 'abc'], dtype=StringDType())

The `np.char` / `np.strings` module has vectorized string functions:

In [16]:
names = np.array(['alice', 'bob', 'carol'])
print(np.strings.upper(names))
print(np.strings.str_len(names))

['ALICE' 'BOB' 'CAROL']
[5 3 5]


### What do `<` and `|` mean in `<U4` or `|S3`?

They describe byte order: `<` is little-endian, `>` big-endian, and `|` means it doesn't apply (single bytes). You rarely need to care unless you're reading raw binary files from another system.

- https://numpy.org/doc/stable/reference/generated/numpy.dtype.byteorder.html
- https://en.wikipedia.org/wiki/Endianness

# 1. Creating Arrays

There are a lot of ways to make an array. This notebook goes through the ones you'll actually use day to day.

In [1]:
import numpy as np

## From Python sequences

`np.array` accepts lists, tuples, ranges and nested combinations of them. Sets and dicts don't work, since they have no defined order.

In [2]:
print(np.array([1, 2, 3, 4, 5]))
print(np.array((1, 2, 3, 4, 5)))
print(np.array(range(5)))

[1 2 3 4 5]
[1 2 3 4 5]
[0 1 2 3 4]


In [3]:
# nested lists give you a 2-D array
np.array([[1, 2, 3], [4, 5, 6]])

array([[1, 2, 3],
       [4, 5, 6]])

In [4]:
# a set won't give you what you want: you get a 0-d object array
np.array({1, 2, 3})

array({1, 2, 3}, dtype=object)

If you have a generator or some other iterable, use `np.fromiter`:

In [5]:
np.fromiter((x * x for x in range(6)), dtype=int)

array([ 0,  1,  4,  9, 16, 25])

## With a specific dtype

In [6]:
arr = np.array([[1, 2, 3], [4, 5, 6]], dtype='i2')
print(arr)
print('dtype:', arr.dtype)

[[1 2 3]
 [4 5 6]]
dtype: int16


## Ranges of numbers

`np.arange(start, stop, step)` works like Python's `range` but returns an array. `stop` is exclusive.

In [7]:
np.arange(0, 20, 2)

array([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])

Be careful with float steps. Because of rounding, you can't always predict how many elements you'll get. When you care about the count, use `linspace` instead.

In [8]:
print(np.arange(1, 1.3, 0.1))   # 4 elements, not 3
print(len(np.arange(0, 1, 0.1)))

[1.  1.1 1.2 1.3]
10


## Evenly spaced numbers

`np.linspace(start, stop, num, endpoint=True, retstep=False)`

- `start`, `stop`: the interval. `stop` is included unless `endpoint=False`
- `num`: how many values you want (default 50)
- `retstep`: also return the spacing between values

In [9]:
arr, step = np.linspace(0, 5, 8, endpoint=False, retstep=True)
print(arr)
print('step size:', step)

[0.    0.625 1.25  1.875 2.5   3.125 3.75  4.375]
step size: 0.625


In [10]:
np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

`np.logspace` and `np.geomspace` do the same thing on a log scale. Useful for things like learning-rate grids:

In [11]:
print(np.logspace(-3, 0, 4))      # 10**-3 ... 10**0
print(np.geomspace(1, 1000, 4))   # same idea, but you give the endpoints directly

[0.001 0.01  0.1   1.   ]
[   1.   10.  100. 1000.]


## Random values

The recommended way is to create a `Generator` and draw from it. Passing a seed makes the results reproducible. The [Sampling Methods](3.%20Sampling%20Methods.ipynb) notebook goes into much more detail.

In [12]:
rng = np.random.default_rng(42)
rng.random((3, 3))     # uniform on [0, 1)

array([[0.77395605, 0.43887844, 0.85859792],
       [0.69736803, 0.09417735, 0.97562235],
       [0.7611397 , 0.78606431, 0.12811363]])

## Zeros, ones, empty and constant arrays

Each of these comes with a `*_like` version that copies the shape and dtype from an existing array.

| Function | Fills with |
|---|---|
| `np.zeros` / `np.zeros_like` | 0 |
| `np.ones` / `np.ones_like` | 1 |
| `np.full` / `np.full_like` | any constant |
| `np.empty` / `np.empty_like` | whatever happened to be in memory |

In [13]:
np.zeros((2, 3))

array([[0., 0., 0.],
       [0., 0., 0.]])

In [14]:
arr = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.complex64)
zeros = np.zeros_like(arr)
print(zeros)
print('dtype:', zeros.dtype)

[[0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
dtype: complex64


In [15]:
np.ones((3, 2), dtype=int)

array([[1, 1],
       [1, 1],
       [1, 1]])

In [16]:
np.full((3, 3), 7)

array([[7, 7, 7],
       [7, 7, 7],
       [7, 7, 7]])

In [17]:
arr = np.array([[1, 2], [3, 4]], dtype=np.float64)
np.full_like(arr, 5)

array([[5., 5.],
       [5., 5.]])

`np.empty` doesn't initialise anything, so the contents are leftover memory, not zeros. It's slightly faster, but only use it when you're about to overwrite every element anyway.

In [18]:
buf = np.empty((2, 3))
buf[:] = 1.5     # fill it before you read it
buf

array([[1.5, 1.5, 1.5],
       [1.5, 1.5, 1.5]])

## Repeating things

- `np.repeat(a, repeats, axis=None)` repeats each **element**. With no axis, the input gets flattened first.
- `np.tile(a, reps)` repeats the **whole array**. `reps` can be a tuple to tile along several axes.

In [19]:
arr = np.array([[0, 1, 2], [3, 4, 5]])
np.repeat(arr, 2)

array([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5])

In [20]:
# repeat each row
np.repeat(arr, 2, axis=0)

array([[0, 1, 2],
       [0, 1, 2],
       [3, 4, 5],
       [3, 4, 5]])

In [21]:
# repeat each column
np.repeat(arr, 2, axis=1)

array([[0, 0, 1, 1, 2, 2],
       [3, 3, 4, 4, 5, 5]])

In [22]:
# different counts per element
np.repeat([1, 2, 3], [3, 0, 1])

array([1, 1, 1, 3])

In [23]:
np.tile([0, 1, 2], 3)

array([0, 1, 2, 0, 1, 2, 0, 1, 2])

In [24]:
np.tile([0, 1, 2], (2, 2))

array([[0, 1, 2, 0, 1, 2],
       [0, 1, 2, 0, 1, 2]])

## Identity and diagonal matrices

- `np.eye(N, M=None, k=0)`: ones on a diagonal. `k` shifts the diagonal up (positive) or down (negative), and `M` lets you make it non-square.
- `np.identity(n)`: always a square identity matrix, no extra options.

In [25]:
np.eye(4)

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

In [26]:
np.eye(4, k=-1)

array([[0., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.]])

In [27]:
np.eye(3, 5, k=1)

array([[0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.]])

In [28]:
np.identity(3, dtype=int)

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]])

`np.diag` goes both ways: give it a 2-D array and it pulls out the diagonal, give it a 1-D array and it builds a diagonal matrix.

In [29]:
arr = np.arange(9).reshape(3, 3)
print(arr)
print('diagonal:', np.diag(arr))

[[0 1 2]
 [3 4 5]
 [6 7 8]]
diagonal: [0 4 8]


In [30]:
# doesn't have to be square
arr = np.arange(12).reshape(4, 3)
print(arr)
print('diagonal:', np.diag(arr))

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
diagonal: [0 4 8]


In [31]:
np.diag([1, 2, 3, 4])

array([[1, 0, 0, 0],
       [0, 2, 0, 0],
       [0, 0, 3, 0],
       [0, 0, 0, 4]])

## Building arrays from a function or a grid

`np.fromfunction` calls your function with the index arrays, so you can describe each element by its position:

In [32]:
np.fromfunction(lambda i, j: i * 10 + j, (3, 4), dtype=int)

array([[ 0,  1,  2,  3],
       [10, 11, 12, 13],
       [20, 21, 22, 23]])

`np.meshgrid` turns 1-D coordinate vectors into 2-D grids. You'll use this a lot for evaluating a function over a plane or for plotting.

In [33]:
x = np.linspace(-1, 1, 3)
y = np.linspace(0, 2, 3)
X, Y = np.meshgrid(x, y)
print(X)
print(Y)
print(X**2 + Y**2)

[[-1.  0.  1.]
 [-1.  0.  1.]
 [-1.  0.  1.]]
[[0. 0. 0.]
 [1. 1. 1.]
 [2. 2. 2.]]
[[1. 0. 1.]
 [2. 1. 2.]
 [5. 4. 5.]]


# 2. Inspecting an Array

Before doing anything with an array it helps to know what you're dealing with: its shape, dtype and how much memory it takes up.

In [1]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int64)
arr

array([[1, 2, 3],
       [4, 5, 6]])

## Quick summary

`np.info` prints a summary straight away (it returns `None`, so there's no need to wrap it in `print`).

In [2]:
np.info(arr)

class:  ndarray
shape:  (2, 3)
strides:  (24, 8)
itemsize:  8
aligned:  True
contiguous:  True
fortran:  False
data pointer: 0x13ab9400
byteorder:  little
byteswap:  False
type: int64


## The attributes you'll use most

| Attribute | Meaning |
|---|---|
| `arr.dtype` | element type |
| `arr.shape` | size along each dimension |
| `arr.ndim` | number of dimensions |
| `arr.size` | total number of elements |
| `len(arr)` | size of the **first** dimension only |
| `arr.itemsize` | bytes per element |
| `arr.nbytes` | total bytes (`size * itemsize`) |

In [3]:
print('dtype:   ', arr.dtype)
print('shape:   ', arr.shape)
print('ndim:    ', arr.ndim)
print('size:    ', arr.size)
print('len:     ', len(arr))
print('itemsize:', arr.itemsize)
print('nbytes:  ', arr.nbytes)

dtype:    int64
shape:    (2, 3)
ndim:     2
size:     6
len:      2
itemsize: 8
nbytes:   48


`len` vs `size` trips people up. `len` only looks at the first axis:

In [4]:
cube = np.zeros((4, 5, 6))
len(cube), cube.size

(4, 120)

## How much memory does the dtype save?

In [5]:
for dt in ['int8', 'int32', 'float64']:
    a = np.zeros(1_000_000, dtype=dt)
    print(f'{dt:>8}: {a.nbytes / 1e6:.0f} MB')

    int8: 1 MB
   int32: 4 MB
 float64: 8 MB


## Strides and memory layout

`strides` tells you how many bytes to jump to move one step along each axis. You don't need it often, but it explains why some operations are free (transposing, slicing) and others need a copy.

In [6]:
print(arr.strides)       # 24 bytes to the next row, 8 to the next column
print(arr.T.strides)     # transposing just swaps the strides, no data moves

(24, 8)
(8, 24)


In [7]:
print(arr.flags['C_CONTIGUOUS'], arr.T.flags['C_CONTIGUOUS'])

True False


## Looking at the values

A few quick sanity checks that are worth running on any new data:

In [8]:
rng = np.random.default_rng(0)
data = rng.normal(loc=10, scale=2, size=(1000, 3))

print('min: ', data.min(axis=0))
print('max: ', data.max(axis=0))
print('mean:', data.mean(axis=0))
print('any NaN?', np.isnan(data).any())

min:  [2.45544969 2.20115654 3.77098159]
max:  [16.13207348 15.94514859 15.69890992]
mean: [ 9.79691519 10.05685054  9.94240763]
any NaN? False


For large arrays you can control how much gets printed with `np.set_printoptions` or, temporarily, with the `np.printoptions` context manager:

In [9]:
with np.printoptions(precision=2, suppress=True, threshold=10):
    print(data)

[[10.25  9.74 11.28]
 [10.21  8.93 10.72]
 [12.61 11.89  8.59]
 ...
 [10.1  12.77  8.7 ]
 [ 7.62  6.94 11.99]
 [ 6.68  6.01  8.77]]


# 3. Random Sampling

NumPy has two random APIs:

1. **The `Generator` API** (`np.random.default_rng()`). This is what you should use in new code. It's faster, has better statistical properties, and doesn't rely on global state.
2. **The legacy API** (`np.random.rand`, `np.random.seed`, ...). It still works and you'll see it all over older code and tutorials, so it's covered at the end.

## Creating a generator

Pass a seed if you want the same numbers every time you run the notebook. Two generators with the same seed produce identical streams, and they don't interfere with each other.

In [2]:
rng = np.random.default_rng(123)
rng.random(3)

array([0.68235186, 0.05382102, 0.22035987])

In [3]:
a = np.random.default_rng(7)
b = np.random.default_rng(7)
a.random(3), b.random(3)

(array([0.62509547, 0.8972138 , 0.77568569]),
 array([0.62509547, 0.8972138 , 0.77568569]))

## Uniform floats on [0, 1)

In [4]:
print(rng.random())          # a single float
print(rng.random(3))         # 1-D
print(rng.random((2, 3)))    # 2-D, note the shape is a tuple

0.1843718106986697
[0.1759059  0.81209451 0.923345  ]
[[0.2765744  0.81975456 0.88989269]
 [0.51297046 0.2449646  0.8242416 ]]


For a different range use `uniform(low, high, size)`:

In [5]:
rng.uniform(-5, 5, size=4)

array([-2.86237037,  2.41467052,  1.29940205,  4.27407259])

## Integers

`integers(low, high, size)` samples from `[low, high)`. Pass `endpoint=True` if you want `high` included.

In [6]:
print(rng.integers(1, 10, size=5))
print(rng.integers(1, 6, size=10, endpoint=True))   # roll a die ten times

[7 3 8 8 2]
[4 5 2 2 1 1 3 1 4 3]


## Common distributions

In [7]:
print(rng.standard_normal((2, 3)))                 # mean 0, std 1
print(rng.normal(loc=170, scale=10, size=5))       # heights in cm, say
print(rng.binomial(n=10, p=0.5, size=5))           # heads in 10 coin flips
print(rng.poisson(lam=3, size=5))
print(rng.exponential(scale=2.0, size=3))

[[-0.14597789  1.28190223  1.07403062]
 [ 0.39262084  0.00511431 -0.36176687]]
[157.69767805 182.26229293 148.27956113 166.29852654 171.6438007 ]
[6 7 4 5 3]
[3 1 0 2 1]
[1.35098514 1.53062042 1.98582649]


A quick check that the normal samples look right:

In [8]:
x = rng.normal(loc=5, scale=2, size=100_000)
print(round(x.mean(), 2), round(x.std(), 2))

5.0 2.0


## Picking from an existing array

`choice(a, size, replace=True, p=None)`

- `a` can be an array or an int (`n` means `np.arange(n)`)
- `replace=False` samples without replacement
- `p` gives a probability for each element (must sum to 1)

In [9]:
print(rng.choice(['red', 'green', 'blue'], size=8))
print(rng.choice(3, size=10, p=[0.1, 0.8, 0.1]))
print(rng.choice(10, size=5, replace=False))

['red' 'blue' 'red' 'red' 'red' 'blue' 'green' 'green']
[2 1 1 1 1 1 1 1 1 1]
[5 2 4 8 1]


## Shuffling

- `shuffle` changes the array **in place** and returns `None`
- `permutation` returns a shuffled **copy**
- `permuted` shuffles each row (or column) independently

For 2-D arrays, `shuffle` and `permutation` only reorder along the first axis, so rows stay intact.

In [10]:
arr = np.arange(10)
rng.shuffle(arr)
arr

array([1, 9, 2, 8, 7, 3, 4, 6, 0, 5])

In [11]:
arr = np.arange(10)
print('original:', arr)
print('permuted:', rng.permutation(arr))
print('original is unchanged:', arr)

original: [0 1 2 3 4 5 6 7 8 9]
permuted: [8 2 4 0 1 7 6 3 9 5]
original is unchanged: [0 1 2 3 4 5 6 7 8 9]


In [12]:
grid = np.arange(12).reshape(3, 4)
print(rng.permutation(grid))           # rows move as whole units
print(rng.permuted(grid, axis=1))      # each row gets shuffled separately

[[ 4  5  6  7]
 [ 8  9 10 11]
 [ 0  1  2  3]]
[[ 3  1  0  2]
 [ 4  7  6  5]
 [ 9 10  8 11]]


### Example: a train/test split

A typical use of `permutation`, shuffling indices and slicing them:

In [13]:
X = np.arange(20).reshape(10, 2)
y = np.arange(10)

idx = rng.permutation(len(X))
cut = int(0.8 * len(X))
train_idx, test_idx = idx[:cut], idx[cut:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(X_train.shape, X_test.shape)
print(y_test)

(8, 2) (2, 2)
[0 5]


## The legacy API

You'll run into these in older code. They all draw from one hidden global generator, which is why `np.random.seed` affects everything, including code in libraries you call. That makes it easy to get results that aren't actually reproducible.

| Legacy | Generator equivalent |
|---|---|
| `np.random.seed(s)` | `rng = np.random.default_rng(s)` |
| `np.random.rand(3, 3)` | `rng.random((3, 3))` |
| `np.random.randn(3, 3)` | `rng.standard_normal((3, 3))` |
| `np.random.randint(lo, hi, n)` | `rng.integers(lo, hi, n)` |
| `np.random.random_sample(n)` | `rng.random(n)` |
| `np.random.choice(...)` | `rng.choice(...)` |
| `np.random.shuffle(a)` | `rng.shuffle(a)` |
| `np.random.permutation(a)` | `rng.permutation(a)` |

If you need an isolated legacy generator, `np.random.RandomState(seed)` gives you one.

In [14]:
np.random.seed(123)
print(np.random.rand(3))
print(np.random.randint(1, 10, 3))

rs = np.random.RandomState(321)    # independent from the global seed
print(rs.rand(3))

[0.69646919 0.28613933 0.22685145]
[7 2 4]
[0.88594794 0.07791236 0.97964616]


# 4. Math and Statistics

Most math in NumPy is **element-wise** and **vectorized**: you write the operation once and it's applied to every element, without a Python loop.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
arr = np.arange(1, 13).reshape(3, 4)
arr

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

## A note on `axis` (read this first)

Reductions like `sum`, `mean` and `max` take an `axis` argument. The easiest way to think about it: **`axis` is the dimension that gets collapsed**.

- `axis=0` collapses the rows, so you get **one result per column**
- `axis=1` collapses the columns, so you get **one result per row**
- no axis means reduce over everything and get a single number

For our 3×4 array, `axis=0` gives 4 values and `axis=1` gives 3.

In [2]:
print('per column (axis=0):', arr.sum(axis=0))
print('per row    (axis=1):', arr.sum(axis=1))
print('everything:         ', arr.sum())

per column (axis=0): [15 18 21 24]
per row    (axis=1): [10 26 42]
everything:          78


Pass `keepdims=True` to keep the reduced axis around with length 1. That makes it easy to broadcast the result back against the original, e.g. to get each value's share of its row:

In [3]:
row_totals = arr.sum(axis=1, keepdims=True)
print(row_totals.shape)
print(np.round(arr / row_totals, 2))

(3, 1)
[[0.1  0.2  0.3  0.4 ]
 [0.19 0.23 0.27 0.31]
 [0.21 0.24 0.26 0.29]]


## Arithmetic

The usual operators work element-wise, both with scalars and with arrays of the same shape.

In [4]:
print(arr + 10)
print(arr * 2)
print(arr / 4)
print(arr // 4)      # floor division
print(arr % 4)       # remainder
print(arr ** 2)

[[11 12 13 14]
 [15 16 17 18]
 [19 20 21 22]]
[[ 2  4  6  8]
 [10 12 14 16]
 [18 20 22 24]]
[[0.25 0.5  0.75 1.  ]
 [1.25 1.5  1.75 2.  ]
 [2.25 2.5  2.75 3.  ]]
[[0 0 0 1]
 [1 1 1 2]
 [2 2 2 3]]
[[1 2 3 0]
 [1 2 3 0]
 [1 2 3 0]]
[[  1   4   9  16]
 [ 25  36  49  64]
 [ 81 100 121 144]]


In [5]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a + b, a * b, b / a)

[11 22 33] [10 40 90] [10. 10. 10.]


Every operator has a function version (`np.add`, `np.multiply`, ...). The functions accept an `out` argument, so you can write the result into an existing array instead of allocating a new one. The augmented operators (`+=`, `*=`) do the same thing.

In [6]:
a = np.array([1, 2, 3])
np.add(a, [8, 9, 10], out=a)
print(a)

a *= 2
print(a)

[ 9 11 13]
[18 22 26]


## Exponents, logs, roots

In [7]:
x = np.array([1, 2, 4, 8])
print(np.exp(x))
print(np.log(x))      # natural log
print(np.log2(x))
print(np.log10(x))
print(np.sqrt(x))
print(np.power(x, 3))

[2.71828183e+00 7.38905610e+00 5.45981500e+01 2.98095799e+03]
[0.         0.69314718 1.38629436 2.07944154]
[0. 1. 2. 3.]
[0.         0.30103    0.60205999 0.90308999]
[1.         1.41421356 2.         2.82842712]
[  1   8  64 512]


`log` of zero or a negative number gives `-inf` or `nan` with a warning, not an error. If you're working with small probabilities, `np.log1p` and `np.expm1` are more accurate near zero.

In [8]:
with np.errstate(divide='ignore', invalid='ignore'):
    print(np.log(np.array([0.0, -1.0])))
print(np.log1p(1e-10), np.log(1 + 1e-10))

[-inf  nan]
9.999999999500001e-11 1.000000082690371e-10


## Trigonometry

Angles are in radians. Use `np.deg2rad` / `np.rad2deg` to convert.

In [9]:
angles = np.deg2rad([0, 30, 45, 90])
print(np.round(np.sin(angles), 3))
print(np.round(np.cos(angles), 3))
print(np.pi, np.e)

[0.    0.5   0.707 1.   ]
[1.    0.866 0.707 0.   ]
3.141592653589793 2.718281828459045


## Rounding, absolute values and clipping

In [10]:
x = np.array([-2.5, -1.5, -0.4, 0.5, 1.5, 2.567])
print(np.round(x))        # rounds half to even: 0.5 -> 0, 1.5 -> 2
print(np.round(x, 1))
print(np.floor(x))
print(np.ceil(x))
print(np.trunc(x))
print(np.abs(x))
print(np.clip(x, -1, 1))  # limit values to a range

[-2. -2. -0.  0.  2.  3.]
[-2.5 -1.5 -0.4  0.5  1.5  2.6]
[-3. -2. -1.  0.  1.  2.]
[-2. -1. -0.  1.  2.  3.]
[-2. -1. -0.  0.  1.  2.]
[2.5   1.5   0.4   0.5   1.5   2.567]
[-1.  -1.  -0.4  0.5  1.   1. ]


`np.modf` splits floats into their fractional and integer parts:

In [11]:
frac, whole = np.modf(np.array([3.75, -1.25, 10.5]))
print('fractional:', frac)
print('integer:   ', whole)

fractional: [ 0.75 -0.25  0.5 ]
integer:    [ 3. -1. 10.]


## Min, max and where they are

In [12]:
m = rng.integers(0, 100, size=(3, 4))
print(m)
print('max per column:', m.max(axis=0))
print('min per row:   ', m.min(axis=1))
print('overall max:   ', m.max())

[[85 63 51 26]
 [30  4  7  1]
 [17 81 64 91]]
max per column: [85 81 64 91]
min per row:    [26  1 17]
overall max:    91


`argmin` / `argmax` return the **position** instead of the value. With no axis, the index refers to the flattened array, and `np.unravel_index` converts it back to a row/column pair.

In [13]:
print('row of the max in each column:', m.argmax(axis=0))
print('column of the min in each row:', m.argmin(axis=1))

flat = m.argmax()
print('flat index:', flat, '-> (row, col):', np.unravel_index(flat, m.shape))

row of the max in each column: [0 2 2 2]
column of the min in each row: [3 3 0]
flat index: 11 -> (row, col): (np.int64(2), np.int64(3))


`np.maximum` / `np.minimum` compare two arrays element by element. Not the same as `max`/`min`, which reduce one array.

In [14]:
a = np.array([1, 3, 5, 7, 9])
b = np.array([0, 4, 3, 8, 7])
print(np.maximum(a, b))
print(np.minimum(a, b))
print(np.maximum(a, 4))   # handy for things like ReLU

[1 4 5 8 9]
[0 3 3 7 7]
[4 4 5 7 9]


## Descriptive statistics

In [15]:
scores = rng.normal(70, 10, size=(5, 3)).round(1)   # 5 students, 3 exams
scores

array([[83. , 79.5, 63. ],
       [57.3, 63.8, 70.4],
       [46.7, 67.8, 57.5],
       [62.7, 64.6, 66.8],
       [74.1, 80.4, 68.7]])

In [16]:
print('mean per exam:   ', scores.mean(axis=0).round(2))
print('median per exam: ', np.median(scores, axis=0))
print('std per exam:    ', scores.std(axis=0).round(2))
print('var per exam:    ', scores.var(axis=0).round(2))
print('mean per student:', scores.mean(axis=1).round(2))

mean per exam:    [64.76 71.22 65.28]
median per exam:  [62.7 67.8 66.8]
std per exam:     [12.7   7.26  4.6 ]
var per exam:     [161.2   52.68  21.19]
mean per student: [75.17 63.83 57.33 64.7  74.4 ]


Note that `std` and `var` default to the **population** formula (dividing by N). For the sample version (N-1), which is what most stats packages use, pass `ddof=1`:

In [17]:
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print(x.std(), x.std(ddof=1))

2.0 2.138089935299395


In [18]:
x = rng.random(1000)
print(np.percentile(x, [5, 50, 95]))
print(np.quantile(x, [0.05, 0.5, 0.95]))   # same thing, but 0-1 instead of 0-100

[0.06620677 0.52609514 0.9452206 ]
[0.06620677 0.52609514 0.9452206 ]


A weighted average:

In [19]:
grades = np.array([80, 90, 70])
weights = np.array([0.2, 0.5, 0.3])
np.average(grades, weights=weights)

np.float64(82.0)

## Covariance and correlation

`np.cov` and `np.corrcoef` treat each **row** as a variable by default. If your variables are in columns (the usual layout for data), pass `rowvar=False`.

In [20]:
height = rng.normal(170, 10, 200)
weight = 0.5 * height + rng.normal(0, 5, 200)
data = np.column_stack([height, weight])    # shape (200, 2), variables in columns

print(np.cov(data, rowvar=False).round(1))
print(np.corrcoef(data, rowvar=False).round(3))

[[95.9 54.6]
 [54.6 59.3]]
[[1.    0.724]
 [0.724 1.   ]]


## Cumulative sums and products

In [21]:
a = np.arange(1, 7).reshape(2, 3)
print(a)
print(np.cumsum(a))            # flattened
print(np.cumsum(a, axis=0))    # running total down each column
print(np.cumsum(a, axis=1))    # running total along each row
print(np.cumprod(a, axis=1))

[[1 2 3]
 [4 5 6]]
[ 1  3  6 10 15 21]
[[1 2 3]
 [5 7 9]]
[[ 1  3  6]
 [ 4  9 15]]
[[  1   2   6]
 [  4  20 120]]


`np.diff` goes the other way, giving the differences between neighbours:

In [22]:
prices = np.array([100, 102, 101, 105, 110])
print(np.diff(prices))
print(np.diff(prices) / prices[:-1])   # daily returns

[ 2 -1  4  5]
[ 0.02       -0.00980392  0.03960396  0.04761905]


## Comparisons and boolean logic

Comparisons give boolean arrays. Combine them with `&`, `|` and `~` (not `and`/`or`/`not`), and **always wrap each condition in parentheses** because `&` binds tighter than `<`.

In [23]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([5, 4, 3, 2, 1])
print(a == b)
print(a < 3)
print((a > 1) & (a < 5))
print((a < 2) | (a > 4))
print(~(a > 2))

[False False  True False False]
[ True  True False False False]
[False  True  True  True False]
[ True False False False  True]
[ True  True False False False]


`True` counts as 1, so `sum` counts matches and `mean` gives the fraction:

In [24]:
x = rng.normal(size=1000)
print('values above 1:', (x > 1).sum())
print('fraction above 1:', (x > 1).mean())
print('any negative?', (x < 0).any(), ' all finite?', np.isfinite(x).all())

values above 1: 156
fraction above 1: 0.156
any negative? True  all finite? True


Never compare floats with `==`. Use `np.isclose` / `np.allclose`:

In [25]:
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
print(np.allclose([1.0, 2.0], [1.0, 2.0 + 1e-10]))

False
True
True


# 5. Indexing and Slicing

There are three kinds of indexing in NumPy:

1. **Basic** indexing with integers and slices. Returns a **view** of the original data.
2. **Boolean** indexing with a mask. Returns a **copy**.
3. **Fancy** indexing with lists/arrays of integers. Also returns a **copy**.

The view-vs-copy difference matters a lot once you start modifying things. There's a whole notebook on it ([12. Views and Copies](12.%20Views%20and%20Copies.ipynb)).

In [1]:
import numpy as np

arr = np.arange(100).reshape(10, 10)
arr

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

## Single elements

In [2]:
print(arr[5, 5])
print(arr[5][5])    # works too, but creates a temporary row first, so prefer arr[5, 5]
print(arr[-1, -1])  # negative indices count from the end

55
55
99


## Slicing

The syntax is `start:stop:step` in each dimension. `stop` is exclusive, and anything you leave out takes its default (start of axis, end of axis, step 1).

In [3]:
arr[1:3, 4:6]

array([[14, 15],
       [24, 25]])

In [4]:
print(arr[0])          # first row
print(arr[:, 0])       # first column
print(arr[::3, ::3])   # every third row and column
print(arr[::-1, 0])    # first column reversed

[0 1 2 3 4 5 6 7 8 9]
[ 0 10 20 30 40 50 60 70 80 90]
[[ 0  3  6  9]
 [30 33 36 39]
 [60 63 66 69]
 [90 93 96 99]]
[90 80 70 60 50 40 30 20 10  0]


Notice that `arr[:, 0]` gives a 1-D array. If you want to keep it as a column (shape `(10, 1)`), slice instead of indexing: `arr[:, 0:1]`, or `arr[:, [0]]`.

In [5]:
print(arr[:, 0].shape, arr[:, 0:1].shape)

(10,) (10, 1)


`...` (ellipsis) stands for "as many `:` as needed". It's mostly useful with higher-dimensional arrays.

In [6]:
cube = np.arange(16).reshape(2, 2, 2, 2)
print(np.array_equal(cube[0, ...], cube[0, :, :, :]))
print(cube[..., 0].shape)    # last axis fixed, all others kept

True
(2, 2, 2)


## Assigning to a slice

A scalar on the right gets broadcast to fill the whole slice:

In [7]:
grid = np.zeros((5, 6), dtype=int)
grid[1:3, :] = 1       # rows 1 and 2
grid[:, 4:] = 9        # last two columns
grid

array([[0, 0, 0, 0, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [0, 0, 0, 0, 9, 9],
       [0, 0, 0, 0, 9, 9]])

Since slices are views, assigning through them changes the original array:

In [8]:
row = grid[0]
row[:] = 7
grid

array([[7, 7, 7, 7, 7, 7],
       [1, 1, 1, 1, 9, 9],
       [1, 1, 1, 1, 9, 9],
       [0, 0, 0, 0, 9, 9],
       [0, 0, 0, 0, 9, 9]])

## Boolean indexing

Index with a boolean array of the same length and you get the elements where it's `True`. The result is always a copy.

In [9]:
a = np.arange(25).reshape(5, 5)
keep = np.array([True, True, False, True, False])
a[keep]          # rows 0, 1 and 3

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [15, 16, 17, 18, 19]])

In [10]:
a[~keep]         # the other rows

array([[10, 11, 12, 13, 14],
       [20, 21, 22, 23, 24]])

Usually the mask comes from a condition on the data itself:

In [11]:
print(a[a > 20])                      # flattened result
print(a[(a % 2 == 0) & (a < 10)])     # even numbers below 10

[21 22 23 24]
[0 2 4 6 8]


Or from a condition on another array, for example picking the rows of a table based on one column:

In [12]:
ages = np.array([23, 35, 17, 52, 41])
salaries = np.array([30, 55, 0, 80, 62])
people = np.column_stack([ages, salaries])

people[(ages >= 18) & (salaries > 50)]

array([[35, 55],
       [52, 80],
       [41, 62]])

Boolean masks are also the simplest way to replace values:

In [13]:
b = a.copy()
b[b > 20] = 0
b

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19],
       [20,  0,  0,  0,  0]])

## Fancy indexing

Pass arrays of integers to pick arbitrary elements in any order.

In [14]:
a = np.arange(10, 20)
a[[3, 0, 3, 7]]      # repeats are fine

array([13, 10, 13, 17])

With 2-D arrays, the index arrays are **paired up**: `a[[r0, r1], [c0, c1]]` gives `a[r0, c0]` and `a[r1, c1]`, not a 2×2 block.

In [15]:
m = np.arange(100).reshape(10, 10)
m[[3, 1, 2], [3, 2, 1]]    # m[3,3], m[1,2], m[2,1]

array([33, 12, 21])

To get a **block** made of some rows and some columns, use `np.ix_`, or index in two steps:

In [16]:
rows, cols = [3, 1, 2], [6, 4, 8]
print(m[np.ix_(rows, cols)])
print(m[rows][:, cols])    # same result, but makes an extra temporary copy

[[36 34 38]
 [16 14 18]
 [26 24 28]]
[[36 34 38]
 [16 14 18]
 [26 24 28]]


## Finding elements by condition

- `np.where(cond)` returns the **indices** where `cond` is true (one array per dimension)
- `np.where(cond, x, y)` picks from `x` where true and `y` where false
- `np.argwhere(cond)` returns the indices as one `(n, ndim)` array
- `np.nonzero(a)` is the same as `np.where(a != 0)`

In [17]:
a = np.arange(16).reshape(4, 4)
print(np.where(a > 12))
print(np.argwhere(a > 12))

(array([3, 3, 3]), array([1, 2, 3]))
[[3 1]
 [3 2]
 [3 3]]


In [18]:
np.where(a > 5, -1, a)    # replace values above 5 with -1, keep the rest

array([[ 0,  1,  2,  3],
       [ 4,  5, -1, -1],
       [-1, -1, -1, -1],
       [-1, -1, -1, -1]])

In [19]:
temps = np.array([12, 25, 31, 18, 35])
np.where(temps > 30, 'hot', np.where(temps > 20, 'warm', 'cool'))

array(['cool', 'warm', 'hot', 'cool', 'hot'], dtype='<U4')

For more than two choices, `np.select` is easier to read than nested `where`:

In [20]:
conditions = [temps > 30, temps > 20]
choices = ['hot', 'warm']
np.select(conditions, choices, default='cool')

array(['cool', 'warm', 'hot', 'cool', 'hot'], dtype='<U4')

## Letting NumPy work out one dimension

Use `-1` for one dimension in `reshape` and NumPy figures it out from the total size:

In [21]:
print(np.arange(16).reshape(4, -1).shape)
print(np.arange(16).reshape(-1, 2).shape)

(4, 4)
(8, 2)


# 6. Sorting

- `np.sort(a)` returns a sorted **copy**
- `a.sort()` sorts **in place** and returns `None`
- `np.argsort(a)` returns the indices that would sort the array

In [1]:
import numpy as np

rng = np.random.default_rng(1)
arr = rng.integers(0, 50, size=(4, 5))
arr

array([[23, 25, 37, 47,  1],
       [ 7, 41, 47, 12, 15],
       [43, 21, 13, 41, 12],
       [20, 32, 27,  4,  1]])

## Sorting along an axis

The default is `axis=-1`, the last axis, so for a 2-D array each **row** gets sorted on its own. `axis=0` sorts each **column**. And `axis=None` flattens first.

Note that each row or column is sorted independently, so the values in a row don't stay together when you sort by column. If you want to sort whole rows by one column, see the `argsort` section below.

In [2]:
print(np.sort(arr))            # each row sorted
print(np.sort(arr, axis=0))    # each column sorted
print(np.sort(arr, axis=None)) # everything, flattened

[[ 1 23 25 37 47]
 [ 7 12 15 41 47]
 [12 13 21 41 43]
 [ 1  4 20 27 32]]
[[ 7 21 13  4  1]
 [20 25 27 12  1]
 [23 32 37 41 12]
 [43 41 47 47 15]]
[ 1  1  4  7 12 12 13 15 20 21 23 25 27 32 37 41 41 43 47 47]


In [3]:
b = arr.copy()
result = b.sort(axis=1)
print(result)    # None! sort() works in place
print(b)

None
[[ 1 23 25 37 47]
 [ 7 12 15 41 47]
 [12 13 21 41 43]
 [ 1  4 20 27 32]]


## Descending order

There's no `reverse=True` option. Reverse the result, or for numbers, sort the negatives:

In [4]:
x = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(np.sort(x)[::-1])
print(-np.sort(-x))

[9 6 5 4 3 2 1 1]
[9 6 5 4 3 2 1 1]


## `argsort`: sort one thing by another

`argsort` is more useful than it looks. Once you have the order, you can apply it to other arrays too.

In [5]:
names = np.array(['Dana', 'Ali', 'Chen', 'Bo'])
scores = np.array([88, 95, 72, 91])

order = np.argsort(scores)[::-1]     # highest score first
for name, score in zip(names[order], scores[order]):
    print(f'{name:5} {score}')

Ali   95
Bo    91
Dana  88
Chen  72


Sorting the rows of a table by one column:

In [6]:
print(arr)
print(arr[arr[:, 2].argsort()])   # rows ordered by the third column

[[23 25 37 47  1]
 [ 7 41 47 12 15]
 [43 21 13 41 12]
 [20 32 27  4  1]]
[[43 21 13 41 12]
 [20 32 27  4  1]
 [23 25 37 47  1]
 [ 7 41 47 12 15]]


## Sorting by several keys

`np.lexsort` sorts by multiple keys. The **last** key you pass is the primary one, which is easy to get backwards.

In [7]:
last  = np.array(['Smith', 'Jones', 'Smith', 'Adams'])
first = np.array(['Zoe',   'Amy',   'Bob',   'Cal'])

idx = np.lexsort((first, last))   # by last name, then first name
np.column_stack((last[idx], first[idx]))

array([['Adams', 'Cal'],
       ['Jones', 'Amy'],
       ['Smith', 'Bob'],
       ['Smith', 'Zoe']], dtype='<U5')

## Top-k without a full sort

If you only need the k smallest (or largest) values, `np.partition` is faster than sorting everything. It guarantees the k-th element is in its sorted position, with smaller values before it and larger ones after, but those two groups aren't sorted themselves.

In [8]:
x = rng.integers(0, 1000, size=20)
k = 3
print(np.partition(x, k)[:k])           # 3 smallest, in no particular order
print(np.sort(np.partition(x, -k)[-k:]))  # 3 largest, sorted afterwards

[123 124 134]
[865 903 976]


In [9]:
idx = np.argpartition(x, -k)[-k:]       # positions of the 3 largest
idx, x[idx]

(array([ 0, 16, 12]), array([865, 903, 976]))

## Searching in a sorted array

`np.searchsorted` finds where values would need to be inserted to keep the array sorted. It's a binary search, so it's fast even on huge arrays, and it's handy for binning:

In [10]:
edges = np.array([0, 18, 65, 120])
ages = np.array([5, 18, 30, 70])
labels = np.array(['child', 'adult', 'senior'])
labels[np.searchsorted(edges, ages, side='right') - 1]

array(['child', 'adult', 'adult', 'senior'], dtype='<U6')

# 7. Reshaping and Manipulating Arrays

Changing the shape, order of axes, or contents of an array.

In [1]:
import numpy as np

arr = np.arange(12).reshape(3, 4)
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

## Transposing

`arr.T`, `arr.transpose()` and `np.transpose(arr)` all do the same thing. They return a **view**, so no data is copied, and changes to the transpose show up in the original.

In [2]:
print(arr.T)
print(np.shares_memory(arr, arr.T))

[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
True


For arrays with more than 2 dimensions, you can give the new order of the axes. Remember that `transpose` doesn't modify the array, so you need to keep the result:

In [3]:
a = np.arange(24).reshape(2, 3, 4)
b = a.transpose(1, 0, 2)
print(a.shape, '->', b.shape)

(2, 3, 4) -> (3, 2, 4)


`swapaxes` swaps two axes, and `moveaxis` moves one axis to a new position. A common case is converting images between channels-last `(H, W, C)` and channels-first `(C, H, W)`:

In [4]:
img = np.zeros((32, 48, 3))     # height, width, channels
print(img.swapaxes(0, 2).shape)
print(np.moveaxis(img, -1, 0).shape)

(3, 48, 32)
(3, 32, 48)


## Reshaping

`reshape` returns a new array with the same data and a different shape. The total number of elements must stay the same. It returns a view when it can, and a copy when it has to.

In [5]:
print(arr.reshape(2, 6))
print(arr.reshape(-1))        # -1 means "work it out"
print(arr.reshape(4, -1).shape)

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
(4, 3)


In [6]:
try:
    arr.reshape(5, 3)
except ValueError as e:
    print('ValueError:', e)

ValueError: cannot reshape array of size 12 into shape (5,3)


By default elements are read and written row by row (`order='C'`). Pass `order='F'` to go column by column:

In [7]:
print(np.arange(6).reshape(2, 3))
print(np.arange(6).reshape(2, 3, order='F'))

[[0 1 2]
 [3 4 5]]
[[0 2 4]
 [1 3 5]]


`np.resize` is different: it doesn't require the size to match. It repeats the data to fill a bigger shape, or cuts it off for a smaller one.

(There's also the in-place method `arr.resize()`, which pads with zeros. It refuses to run if anything else references the array, and in Jupyter that's often the case, so the function is usually less hassle.)

In [8]:
print(np.resize(np.arange(4), (3, 3)))
print(np.resize(np.arange(10), (2, 3)))

[[0 1 2]
 [3 0 1]
 [2 3 0]]
[[0 1 2]
 [3 4 5]]


## Flattening

- `flatten()` always returns a **copy**
- `ravel()` returns a **view** if it can, otherwise a copy. It's faster, but changes may or may not affect the original, so don't rely on either behaviour.

In [9]:
f = arr.flatten()
r = arr.ravel()
print(np.shares_memory(arr, f), np.shares_memory(arr, r))

False True


In [10]:
# the transpose isn't contiguous, so ravel has to copy here
print(np.shares_memory(arr.T, arr.T.ravel()))

False


## Adding and removing dimensions

You often need an extra axis of length 1, for example to turn a 1-D vector into a column so it broadcasts correctly. `np.newaxis` (which is just `None`) and `np.expand_dims` both do this. `squeeze` removes length-1 axes.

In [11]:
v = np.array([1, 2, 3])
print(v.shape)
print(v[:, np.newaxis].shape)    # column
print(v[np.newaxis, :].shape)    # row
print(np.expand_dims(v, axis=0).shape)

(3,)
(3, 1)
(1, 3)
(1, 3)


In [12]:
x = np.zeros((1, 3, 1, 4))
print(x.squeeze().shape)
print(x.squeeze(axis=0).shape)

(3, 4)
(3, 1, 4)


## Flipping and rolling

In [13]:
print(np.flip(arr, axis=0))     # upside down
print(np.flip(arr, axis=1))     # left-right (same as arr[:, ::-1])

[[ 8  9 10 11]
 [ 4  5  6  7]
 [ 0  1  2  3]]
[[ 3  2  1  0]
 [ 7  6  5  4]
 [11 10  9  8]]


In [14]:
print(np.roll(np.arange(6), 2))
print(np.roll(arr, 1, axis=1))

[4 5 0 1 2 3]
[[ 3  0  1  2]
 [ 7  4  5  6]
 [11  8  9 10]]


In [15]:
np.rot90(arr)

array([[ 3,  7, 11],
       [ 2,  6, 10],
       [ 1,  5,  9],
       [ 0,  4,  8]])

## Adding and removing elements

`np.append`, `np.insert` and `np.delete` all return a **new array**. NumPy arrays have a fixed size, so there's no in-place append like with lists.

If you find yourself appending in a loop, collect the values in a Python list and convert once at the end. Appending to an array copies the whole thing every time.

In [16]:
a = np.array([1, 2, 3])
print(np.append(a, 4))
print(np.append(a, [4, 5, 6]))
print(a)    # unchanged

[1 2 3 4]
[1 2 3 4 5 6]
[1 2 3]


Without an `axis`, `append` flattens everything. Pass `axis` to add rows or columns:

In [17]:
m = np.array([[1, 2], [3, 4]])
print(np.append(m, [[5, 6]], axis=0))
print(np.append(m, [[9], [9]], axis=1))

[[1 2]
 [3 4]
 [5 6]]
[[1 2 9]
 [3 4 9]]


In [18]:
# np.insert(array, position, values)
print(np.insert(a, 0, 100))
print(np.insert(a, 1, [7, 8]))
print(np.insert(m, 1, 0, axis=1))   # insert a column of zeros

[100   1   2   3]
[1 7 8 2 3]
[[1 0 2]
 [3 0 4]]


In [19]:
print(np.delete(a, 0))
print(np.delete(a, [0, 2]))
print(np.delete(m, 0, axis=0))      # drop the first row

[2 3]
[2]
[[3 4]]


## Copying

Plain assignment doesn't copy anything, it just gives the same array a second name. Use `.copy()` when you need an independent array.

In [20]:
a = np.array([1, 2, 3])
b = a          # same object
c = a.copy()   # new data

a[0] = 99
print(b, c)

[99  2  3] [1 2 3]


`np.copy(a)` and `np.array(a, copy=True)` do the same thing as `a.copy()`.

One catch: for `object` arrays, `copy()` copies the references, not the objects themselves. If the elements are mutable (lists, dicts), use `copy.deepcopy`.

# 8. Joining and Splitting Arrays

In [1]:
import numpy as np

a = np.array([[1, 2, 3, 4], [1, 2, 3, 4]])
b = np.array([[5, 6, 7, 8], [5, 6, 7, 8]])

## `concatenate`: join along an existing axis

`axis=0` puts `b` below `a` (more rows), `axis=1` puts it to the right (more columns). The arrays must match in every other dimension.

In [2]:
print(np.concatenate((a, b), axis=0))
print(np.concatenate((a, b), axis=1))

[[1 2 3 4]
 [1 2 3 4]
 [5 6 7 8]
 [5 6 7 8]]
[[1 2 3 4 5 6 7 8]
 [1 2 3 4 5 6 7 8]]


## `vstack` / `hstack`

Shortcuts for the two cases above. `vstack` also accepts 1-D arrays and treats them as rows, which `concatenate` won't do.

In [3]:
print(np.vstack((a, b)))
print(np.hstack((a, b)))

[[1 2 3 4]
 [1 2 3 4]
 [5 6 7 8]
 [5 6 7 8]]
[[1 2 3 4 5 6 7 8]
 [1 2 3 4 5 6 7 8]]


In [4]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
print(np.vstack((x, y)))      # two rows
print(np.hstack((x, y)))      # one longer 1-D array
print(np.column_stack((x, y)))  # two columns, often what you actually want

[[1 2 3]
 [4 5 6]]
[1 2 3 4 5 6]
[[1 4]
 [2 5]
 [3 6]]


## `stack`: join along a new axis

`stack` adds a new dimension. All inputs must have the same shape. It's how you'd turn a list of 2-D images into one 3-D batch.

In [5]:
print(np.stack((x, y)).shape)            # (2, 3)
print(np.stack((x, y), axis=1).shape)    # (3, 2)

images = [np.zeros((28, 28)) for _ in range(10)]
print(np.stack(images).shape)

(2, 3)
(3, 2)
(10, 28, 28)


## `np.r_` and `np.c_`

Compact index-style shortcuts. `np.r_` joins along the first axis, `np.c_` along the last. `np.r_` also understands slice syntax, which is handy for building index arrays.

In [6]:
print(np.r_[a, b])
print(np.c_[a, b])
print(np.r_[0:3, 10, 20:23])

[[1 2 3 4]
 [1 2 3 4]
 [5 6 7 8]
 [5 6 7 8]]
[[1 2 3 4 5 6 7 8]
 [1 2 3 4 5 6 7 8]]
[ 0  1  2 10 20 21 22]


## `block`: assemble from pieces

`np.block` builds an array from a nested list of blocks, like writing a block matrix on paper:

In [7]:
I = np.eye(2, dtype=int)
Z = np.zeros((2, 2), dtype=int)
np.block([[I, Z], [Z, 2 * I]])

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 2, 0],
       [0, 0, 0, 2]])

## Splitting

- `np.split(arr, n, axis)` splits into `n` **equal** parts (raises an error if it can't)
- `np.split(arr, [i, j], axis)` splits **at** the given indices
- `np.array_split` is like `split` but allows unequal parts
- `vsplit` / `hsplit` are shortcuts for `axis=0` / `axis=1`

All of them return a list of arrays.

In [8]:
arr = np.arange(36).reshape(6, 6)
top, bottom = np.vsplit(arr, 2)
print(top)
print(bottom)

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]]
[[18 19 20 21 22 23]
 [24 25 26 27 28 29]
 [30 31 32 33 34 35]]


In [9]:
left, mid, right = np.hsplit(arr, 3)
print(mid)

[[ 2  3]
 [ 8  9]
 [14 15]
 [20 21]
 [26 27]
 [32 33]]


In [10]:
# split at positions 1 and 4 along the rows
parts = np.split(arr, [1, 4], axis=0)
[p.shape for p in parts]

[(1, 6), (3, 6), (2, 6)]

In [11]:
try:
    np.split(np.arange(10), 3)
except ValueError as e:
    print('ValueError:', e)

np.array_split(np.arange(10), 3)

ValueError: array split does not result in an equal division


[array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]

### Example: processing in batches

In [12]:
data = np.arange(103)
for i, batch in enumerate(np.array_split(data, 4)):
    print(i, len(batch), batch[:3], '...')

0 26 [0 1 2] ...
1 26 [26 27 28] ...
2 26 [52 53 54] ...
3 25 [78 79 80] ...


# 9. Set Operations

These functions treat arrays as sets. Most of them return **sorted, 1-D** results with duplicates removed.

## Unique values

In [2]:
arr = np.array([3, 1, 2, 3, 3, 1, 4, 5, 6, 2])
np.unique(arr)

array([1, 2, 3, 4, 5, 6])

`np.unique` has a few useful extra outputs:

- `return_counts`: how many times each value appears
- `return_index`: where each value first appears
- `return_inverse`: indices that rebuild the original array from the unique values (basically label-encoding)

In [3]:
values, counts = np.unique(arr, return_counts=True)
print(values)
print(counts)
print('most common:', values[counts.argmax()])

[1 2 3 4 5 6]
[2 2 3 1 1 1]
most common: 3


In [4]:
colors = np.array(['red', 'blue', 'red', 'green', 'blue', 'red'])
labels, codes = np.unique(colors, return_inverse=True)
print(labels)
print(codes)
print(labels[codes])    # back to the original

['blue' 'green' 'red']
[2 0 2 1 0 2]
['red' 'blue' 'red' 'green' 'blue' 'red']


NumPy 2.0 added clearer single-purpose versions: `unique_values`, `unique_counts`, `unique_inverse` and `unique_all`. Note that `unique_values` doesn't promise sorted output.

In [5]:
res = np.unique_counts(colors)
res.values, res.counts

(array(['blue', 'green', 'red'], dtype='<U5'), array([2, 1, 3]))

For 2-D arrays, `axis=0` finds unique **rows**:

In [6]:
pts = np.array([[0, 1], [1, 1], [0, 1], [2, 0]])
np.unique(pts, axis=0)

array([[0, 1],
       [1, 1],
       [2, 0]])

## Intersection, union, difference

In [7]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([3, 4, 5, 6, 7])

print('a & b :', np.intersect1d(a, b))
print('a | b :', np.union1d(a, b))
print('a - b :', np.setdiff1d(a, b))
print('a ^ b :', np.setxor1d(a, b))    # in one or the other, not both

a & b : [3 4 5]
a | b : [1 2 3 4 5 6 7]
a - b : [1 2]
a ^ b : [1 2 6 7]


`intersect1d` can also tell you where the common elements are in each input:

In [8]:
common, ia, ib = np.intersect1d(a, b, return_indices=True)
print(common, ia, ib)

[3 4 5] [2 3 4] [0 1 2]


For more than two arrays, `functools.reduce` does the job:

In [9]:
from functools import reduce
reduce(np.intersect1d, ([1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6]))

array([3, 4])

## Membership: `isin`

`np.isin(a, b)` tells you, for each element of `a`, whether it appears in `b`. The result has the **same shape as `a`**, so you can use it directly as a mask.

(Older code uses `np.in1d`, which always flattened the result. It was deprecated and has been removed in recent NumPy versions, so use `isin`.)

In [10]:
print(np.isin(a, b))
print(a[np.isin(a, b)])
print(a[~np.isin(a, b)])     # or invert=True

[False False  True  True  True]
[3 4 5]
[1 2]


In [11]:
grid = np.arange(12).reshape(3, 4)
np.isin(grid, [1, 5, 11])

array([[False,  True, False, False],
       [False,  True, False, False],
       [False, False, False,  True]])

### Example: filtering by a list of allowed values

In [12]:
products = np.array(['apple', 'kiwi', 'banana', 'mango', 'apple', 'cherry'])
in_stock = ['apple', 'banana', 'cherry']
products[np.isin(products, in_stock)]

array(['apple', 'banana', 'apple', 'cherry'], dtype='<U6')

# 10. Linear Algebra

Most of this lives in `np.linalg`. The examples use small matrices with known answers so you can check the results by hand.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[4., 1.], [2., 3.]])
B = np.array([[1., 2.], [0., 1.]])

## Element-wise vs matrix multiplication

This is the classic beginner mistake: `*` multiplies **element by element**. For matrix multiplication, use `@` (or `np.matmul` / `np.dot`).

In [2]:
print(A * B)    # element-wise
print(A @ B)    # matrix product

[[4. 2.]
 [0. 3.]]
[[4. 9.]
 [2. 7.]]


In [3]:
# these all give the same matrix product for 2-D arrays
print(np.allclose(A @ B, np.matmul(A, B)), np.allclose(A @ B, A.dot(B)))

True True


`@` also works on **stacks** of matrices: the last two dimensions are the matrices, and everything before is treated as a batch. `np.dot` behaves differently for 3-D+ inputs, so stick with `@`.

In [4]:
rng = np.random.default_rng(0)
batch = rng.random((10, 3, 4))
weights = rng.random((4, 2))
(batch @ weights).shape

(10, 3, 2)

## Vector products

In [5]:
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])
print('dot:  ', u @ v)            # same as np.dot(u, v)
print('outer:\n', np.outer(u, v))
print('cross:', np.cross(u, v))

dot:   32.0
outer:
 [[ 4.  5.  6.]
 [ 8. 10. 12.]
 [12. 15. 18.]]
cross: [-3.  6. -3.]


## Norms

In [6]:
x = np.array([3., 4.])
print(np.linalg.norm(x))          # Euclidean length: 5
print(np.linalg.norm(x, ord=1))   # sum of absolute values: 7
print(np.linalg.norm(x, ord=np.inf))
print(np.linalg.norm(A))          # Frobenius norm for a matrix

5.0
7.0
4.0
5.477225575051661


In [7]:
# normalising each row of a matrix to unit length
M = rng.random((3, 2))
M_unit = M / np.linalg.norm(M, axis=1, keepdims=True)
np.linalg.norm(M_unit, axis=1)

array([1., 1., 1.])

## Trace, determinant, rank

Note that `trace` is `np.trace`, not `np.linalg.trace` (there is one now, but `np.trace` is the one you'll see everywhere).

In [8]:
print('trace:', np.trace(A))
print('det:  ', np.linalg.det(A))         # 4*3 - 1*2 = 10
print('rank: ', np.linalg.matrix_rank(A))

singular = np.array([[1., 2.], [2., 4.]])
print('rank of a singular matrix:', np.linalg.matrix_rank(singular))

trace: 7.0
det:   10.000000000000002
rank:  2
rank of a singular matrix: 1


## Inverse and pseudo-inverse

In [9]:
A_inv = np.linalg.inv(A)
print(A_inv)
print(A @ A_inv)    # identity, up to rounding

[[ 0.3 -0.1]
 [-0.2  0.4]]
[[ 1.  0.]
 [-0.  1.]]


Singular matrices don't have an inverse, and `inv` raises an error. The pseudo-inverse `pinv` works for any matrix, including non-square ones.

In [10]:
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)

print(np.linalg.pinv(singular))

LinAlgError: Singular matrix
[[0.04 0.08]
 [0.08 0.16]]


## Solving linear systems

To solve `A x = b`, use `np.linalg.solve`. It's faster and more accurate than computing `inv(A) @ b`.

Example: 4x + y = 9 and 2x + 3y = 13.

In [11]:
b = np.array([9., 13.])
x = np.linalg.solve(A, b)
print(x)
print(np.allclose(A @ x, b))

[1.4 3.4]
True


## Least squares

When there are more equations than unknowns there's usually no exact solution. `lstsq` finds the best fit. Here we fit a line `y = m*x + c` to noisy points:

In [12]:
x = np.linspace(0, 10, 20)
y = 2.5 * x + 1 + rng.normal(0, 1, x.size)

X = np.column_stack([x, np.ones_like(x)])   # design matrix [x, 1]
coef, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
print('slope, intercept:', coef)

slope, intercept: [2.3939 1.7227]


(For simple polynomial fits, `np.polyfit(x, y, deg)` does the same in one line.)

In [13]:
np.polyfit(x, y, 1)

array([2.3939, 1.7227])

## Eigenvalues and eigenvectors

`eig` returns the eigenvalues and a matrix whose **columns** are the eigenvectors.

A real matrix can have complex eigenvalues, so depending on your NumPy version you may get complex results even when the imaginary parts are all zero. Use `.real` if you know they should be real.

In [14]:
w, V = np.linalg.eig(A)
print('eigenvalues:', w.real)
print('eigenvectors (columns):\n', V)

# check A v = lambda v for the first pair
print(np.allclose(A @ V[:, 0], w[0] * V[:, 0]))

eigenvalues: [5. 2.]
eigenvectors (columns):
 [[ 0.7071+0.j -0.4472+0.j]
 [ 0.7071+0.j  0.8944+0.j]]
True


If your matrix is symmetric (covariance matrices, for example), use `eigh`. It's faster, always returns real values, and sorts them in ascending order.

In [15]:
S = np.array([[2., 1.], [1., 2.]])
w, V = np.linalg.eigh(S)
print(w)
print(V)

[1. 3.]
[[-0.7071  0.7071]
 [ 0.7071  0.7071]]


## Matrix decompositions

### QR

In [16]:
M = rng.random((4, 3))
Q, R = np.linalg.qr(M)
print(Q.shape, R.shape)
print(np.allclose(Q @ R, M))
print(np.allclose(Q.T @ Q, np.eye(3)))    # Q has orthonormal columns

(4, 3) (3, 3)
True
True


### SVD

`svd` returns `U`, the singular values `s` (as a 1-D array, largest first), and `Vh`, which is already the **transpose** of V. A common mistake is to transpose it again.

In [17]:
M = rng.random((4, 3))
U, s, Vh = np.linalg.svd(M, full_matrices=False)
print(U.shape, s.shape, Vh.shape)
print(np.allclose(U @ np.diag(s) @ Vh, M))

(4, 3) (3,) (3, 3)
True


Keeping only the largest singular values gives the best low-rank approximation, which is the idea behind PCA and image compression:

In [18]:
k = 1
M_k = U[:, :k] @ np.diag(s[:k]) @ Vh[:k]
print('rank-1 approximation error:', np.linalg.norm(M - M_k).round(4))

rank-1 approximation error: 0.5973


### Cholesky

For symmetric positive-definite matrices:

In [19]:
L = np.linalg.cholesky(S)
print(L)
print(np.allclose(L @ L.T, S))

[[1.4142 0.    ]
 [0.7071 1.2247]]
True


## Matrix powers

In [20]:
# Fibonacci numbers from powers of a 2x2 matrix
F = np.array([[1, 1], [1, 0]])
np.linalg.matrix_power(F, 10)

array([[89, 55],
       [55, 34]])

# 11. Broadcasting

Broadcasting is how NumPy does arithmetic on arrays of **different shapes** without copying data. You've already used it every time you wrote `arr + 1`. Once you understand the rules, a lot of loops just disappear.

## The rules

NumPy compares the shapes **from the right**, one dimension at a time. Two dimensions are compatible if:

1. they're equal, or
2. one of them is 1.

If one array has fewer dimensions, it's treated as if it had extra 1s on the **left**. Dimensions of size 1 get stretched to match the other array.

| A | B | Result |
|---|---|---|
| `(3, 4)` | `()` (scalar) | `(3, 4)` |
| `(3, 4)` | `(4,)` | `(3, 4)` |
| `(3, 4)` | `(3, 1)` | `(3, 4)` |
| `(3, 1)` | `(1, 4)` | `(3, 4)` |
| `(8, 1, 6)` | `(7, 1)` | `(8, 7, 6)` |
| `(3, 4)` | `(3,)` | **error**: 4 vs 3 |

## Scalar with array

In [2]:
a = np.arange(12).reshape(3, 4)
a * 10

array([[  0,  10,  20,  30],
       [ 40,  50,  60,  70],
       [ 80,  90, 100, 110]])

## Row vector with matrix

A shape `(4,)` array lines up with the **last** axis, so it gets added to every row:

In [3]:
row = np.array([100, 200, 300, 400])
a + row

array([[100, 201, 302, 403],
       [104, 205, 306, 407],
       [108, 209, 310, 411]])

## Column vector with matrix

To add something to every column instead, the vector needs shape `(3, 1)`. A plain `(3,)` array won't work, because it gets compared against the 4:

In [4]:
col = np.array([1000, 2000, 3000])
try:
    a + col
except ValueError as e:
    print('ValueError:', e)

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


In [5]:
a + col[:, np.newaxis]     # or col.reshape(-1, 1)

array([[1000, 1001, 1002, 1003],
       [2004, 2005, 2006, 2007],
       [3008, 3009, 3010, 3011]])

## Two vectors make a grid

`(3, 1)` with `(1, 4)` gives `(3, 4)`. This is a neat way to build tables without loops, like a multiplication table:

In [6]:
x = np.arange(1, 10)
x[:, None] * x[None, :]

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9],
       [ 2,  4,  6,  8, 10, 12, 14, 16, 18],
       [ 3,  6,  9, 12, 15, 18, 21, 24, 27],
       [ 4,  8, 12, 16, 20, 24, 28, 32, 36],
       [ 5, 10, 15, 20, 25, 30, 35, 40, 45],
       [ 6, 12, 18, 24, 30, 36, 42, 48, 54],
       [ 7, 14, 21, 28, 35, 42, 49, 56, 63],
       [ 8, 16, 24, 32, 40, 48, 56, 64, 72],
       [ 9, 18, 27, 36, 45, 54, 63, 72, 81]])

`np.broadcast_shapes` tells you what shape you'll get, without doing the computation. Useful for checking your reasoning:

In [7]:
print(np.broadcast_shapes((8, 1, 6), (7, 1)))
print(np.broadcast_shapes((5, 3), (3,)))

(8, 7, 6)
(5, 3)


## Practical examples

### Standardising columns

Subtract each column's mean and divide by its std. `mean(axis=0)` has shape `(n_features,)`, which broadcasts across the rows:

In [8]:
rng = np.random.default_rng(0)
X = rng.normal(loc=[10, 50, -3], scale=[1, 20, 0.5], size=(500, 3))

Z = (X - X.mean(axis=0)) / X.std(axis=0)
print(Z.mean(axis=0).round(6) + 0.0)   # + 0.0 just tidies up -0.0
print(Z.std(axis=0).round(6))

[0. 0. 0.]
[1. 1. 1.]


### Normalising rows

For row-wise operations, use `keepdims=True` so the result stays as a column:

In [9]:
counts = np.array([[1, 3, 6], [2, 2, 0], [5, 0, 5]])
probs = counts / counts.sum(axis=1, keepdims=True)
probs

array([[0.1, 0.3, 0.6],
       [0.5, 0.5, 0. ],
       [0.5, 0. , 0.5]])

### Pairwise distances

Given `n` points and `m` centres, compute every point-to-centre distance at once. Shapes: `(n, 1, 2) - (1, m, 2)` gives `(n, m, 2)`.

In [10]:
points = rng.random((5, 2))
centres = np.array([[0.1, 0.1], [0.9, 0.9], [0.1, 0.9]])

diff = points[:, None, :] - centres[None, :, :]
dist = np.sqrt((diff ** 2).sum(axis=-1))
print(dist.shape)
print(dist.round(3))
print('closest centre for each point:', dist.argmin(axis=1))

(5, 3)
[[0.576 0.722 0.264]
 [0.317 0.854 0.515]
 [0.269 0.87  0.691]
 [0.705 0.609 0.901]
 [0.819 0.711 1.081]]
closest centre for each point: [2 0 0 1 1]


### Applying a colour tint to an image

An RGB image has shape `(H, W, 3)`. Multiplying by a length-3 array scales each channel:

In [11]:
img = np.ones((2, 2, 3))
tint = np.array([1.0, 0.5, 0.2])
(img * tint)[0, 0]

array([1. , 0.5, 0.2])

## Things to watch out for

**Accidental broadcasting.** A shape mismatch that *happens* to be compatible gives you a big matrix instead of an error. This one catches people all the time:

In [12]:
y_true = np.array([1., 2., 3.])
y_pred = np.array([[1.1], [1.9], [3.2]])   # (3, 1), e.g. straight out of a model

err = y_true - y_pred
print(err.shape)    # (3, 3), not (3,)!
print('wrong MSE:', (err ** 2).mean())
print('right MSE:', ((y_true - y_pred.ravel()) ** 2).mean())

(3, 3)
wrong MSE: 1.4200000000000004
right MSE: 0.020000000000000035


**Memory.** Broadcasting itself doesn't copy, but the *result* has the full broadcast shape. The pairwise-distance trick on 100k × 100k points would need 80 GB for the result alone.

If you want a broadcast array without computing anything (as a read-only view), use `np.broadcast_to`:

In [13]:
b = np.broadcast_to(np.arange(3), (4, 3))
print(b)
print(b.strides)    # stride 0 on the first axis: the same row is reused

[[0 1 2]
 [0 1 2]
 [0 1 2]
 [0 1 2]]
(0, 8)


# 12. Views and Copies

This is probably the most common source of confusing bugs in NumPy code. Some operations give you a **view** that shares memory with the original array, others give you an independent **copy**. Modify a view and the original changes too.

## No copy at all: assignment

`b = a` doesn't create a new array. Both names point to the same object.

In [2]:
a = np.arange(5)
b = a
b[0] = 100
print(a)
print(b is a)

[100   1   2   3   4]
True


## Views

A view is a new array object that looks at the same data. Basic slicing returns a view:

In [3]:
a = np.arange(10)
s = a[2:5]
s[:] = -1
print(a)

[ 0  1 -1 -1 -1  5  6  7  8  9]


You can check with `np.shares_memory`, or look at `.base` (a view's `base` is the array it came from; a fresh array has `base = None`):

In [4]:
print(np.shares_memory(a, s))
print(s.base is a)
print(a.base)

True
True
None


## Copies

Explicit `.copy()`, plus boolean and fancy indexing, always give you a copy:

In [5]:
a = np.arange(10)

c = a[2:5].copy()
m = a[a > 5]
f = a[[1, 3, 5]]

c[:] = 0; m[:] = 0; f[:] = 0
print(a)     # untouched

[0 1 2 3 4 5 6 7 8 9]


## Cheat sheet

| Operation | Result |
|---|---|
| `b = a` | same object |
| `a[1:5]`, `a[::2]`, `a[:, 0]` | view |
| `a.T`, `a.transpose()`, `a.swapaxes()` | view |
| `a.reshape(...)` | view if possible, otherwise copy |
| `a.ravel()` | view if possible, otherwise copy |
| `a.view(dtype)` | view |
| `np.newaxis`, `squeeze`, `expand_dims` | view |
| `a.flatten()` | copy |
| `a.copy()`, `np.copy(a)`, `np.array(a)` | copy |
| `a[[0, 2]]` (fancy indexing) | copy |
| `a[a > 0]` (boolean indexing) | copy |
| `a + 1`, `np.sqrt(a)`, ... | new array |
| `a.astype(...)` | copy (by default) |

## Where this bites

### 1. Modifying a slice you thought was separate

In [6]:
data = np.array([3., 1., 4., 1., 5.])
first_three = data[:3]
first_three /= first_three.max()     # meant to normalise just this piece
print(data)                          # ...but the original changed too

[0.75 0.25 1.   1.   5.  ]


Fix: call `.copy()` when you plan to modify something and want the original kept intact.

### 2. Assignment with chained indexing

Assigning through a fancy index followed by a slice doesn't work, because the first step already made a copy:

In [7]:
a = np.zeros((3, 3))
a[[0, 2]][:, 0] = 1    # modifies a temporary copy, not a
print(a)

a[[0, 2], 0] = 1       # do it in one step instead
print(a)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]]


Boolean masks work fine in a single assignment, since NumPy turns `a[mask] = value` into a direct write:

In [8]:
a = np.arange(6)
a[a % 2 == 0] = -1
a

array([-1,  1, -1,  3, -1,  5])

### 3. Keeping a huge array alive

A small view keeps the **whole** original array in memory, because it still points into its buffer. If you only need a small piece of a big array, copy it so the big one can be freed.

In [9]:
big = np.ones(10_000_000)
small_view = big[:10]
small_copy = big[:10].copy()
print(small_view.base is big, small_copy.base is None)

True True


## When views are what you want

Views are fast and memory-efficient, and sometimes they're exactly what you need. For example, a sliding window over a time series without copying anything:

In [10]:
from numpy.lib.stride_tricks import sliding_window_view

x = np.arange(10)
windows = sliding_window_view(x, 3)
print(windows)
print('moving average:', windows.mean(axis=1))
print(np.shares_memory(x, windows))

[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]
 [6 7 8]
 [7 8 9]]
moving average: [1. 2. 3. 4. 5. 6. 7. 8.]
True


`.view(dtype)` reinterprets the same bytes as a different type. It's low level and you won't need it much, but it shows that a view really is the same memory:

In [11]:
f = np.array([1.0], dtype=np.float64)
print(f.view(np.int64))     # the raw bit pattern of 1.0

[4607182418800017408]


# 13. Vectorization and Universal Functions

The main reason to use NumPy is speed, and you only get it if you let NumPy do the looping. This notebook shows why, and the tools for turning loops into array operations.

In [1]:
import numpy as np
import time

rng = np.random.default_rng(0)

## Loops vs vectorized code

A Python loop handles one element at a time, with type checks and function calls on every iteration. A NumPy operation runs a tight compiled loop over the whole buffer.

In [2]:
x = rng.random(1_000_000)

def timeit(fn, repeat=3):
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

def loop_version():
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i] ** 2 + 1
    return out

def numpy_version():
    return x ** 2 + 1

t_loop, t_np = timeit(loop_version), timeit(numpy_version)
print(f'loop:  {t_loop * 1000:.0f} ms')
print(f'numpy: {t_np * 1000:.1f} ms')
print(f'roughly {t_loop / t_np:.0f}x faster')

loop:  244 ms
numpy: 1.2 ms
roughly 198x faster


(Your numbers will differ, but the gap is usually somewhere between 50x and a few hundred x.)

## What's a ufunc?

A **universal function** (ufunc) is a function that works element by element on arrays: `np.add`, `np.sqrt`, `np.maximum`, `np.sin`, and so on. They all support broadcasting, dtype handling and a few extra methods that are worth knowing.

In [3]:
print(type(np.add), np.add.nin, np.add.nout)

<class 'numpy.ufunc'> 2 1


### `reduce`, `accumulate`, `outer`

Every binary ufunc has these methods. `np.sum` is basically `np.add.reduce`, and `np.cumsum` is `np.add.accumulate`, but it works for any ufunc:

In [4]:
a = np.array([3, 1, 4, 1, 5])
print(np.add.reduce(a))
print(np.maximum.accumulate(a))   # running maximum
print(np.multiply.outer([1, 2, 3], [10, 20]))

14
[3 3 4 4 5]
[[10 20]
 [20 40]
 [30 60]]


In [5]:
# running maximum is handy for things like drawdowns
prices = np.array([100, 110, 105, 120, 90, 95])
peak = np.maximum.accumulate(prices)
print('drawdown:', (prices - peak) / peak)

drawdown: [ 0.          0.         -0.04545455  0.         -0.25       -0.20833333]


### `out` and `where`

`out` writes into an existing array (no new allocation). `where` only computes at positions where the mask is true, which is useful for avoiding warnings like division by zero:

In [6]:
num = np.array([1., 2., 3., 4.])
den = np.array([2., 0., 4., 0.])

result = np.zeros_like(num)
np.divide(num, den, out=result, where=den != 0)
result

array([0.5 , 0.  , 0.75, 0.  ])

### `at`: unbuffered in-place operations

With fancy indexing, repeated indices only get applied once when you use `+=`. `np.add.at` applies all of them:

In [7]:
counts = np.zeros(5, dtype=int)
idx = np.array([0, 1, 1, 3, 3, 3])

c1 = counts.copy(); c1[idx] += 1
c2 = counts.copy(); np.add.at(c2, idx, 1)
print(c1)   # repeats collapsed
print(c2)   # actually counted

[1 1 0 1 0]
[1 2 0 3 0]


For counting like this, `np.bincount` is even simpler:

In [8]:
np.bincount(idx, minlength=5)

array([1, 2, 0, 3, 0])

## Turning common loops into array code

### Conditionals → `np.where` / masks

In [9]:
x = rng.normal(size=10)

# loop version
out = []
for v in x:
    out.append(v if v > 0 else 0.1 * v)

# vectorized
vec = np.where(x > 0, x, 0.1 * x)
print(np.allclose(out, vec))

True


### Accumulating in a loop → reductions

In [10]:
values = rng.integers(0, 100, size=1000)

total = 0
for v in values:
    if v % 2 == 0:
        total += v

print(total, values[values % 2 == 0].sum())

24500 24500


### Nested loops → broadcasting

In [11]:
a = rng.random(300)
b = rng.random(400)

def nested():
    out = np.empty((len(a), len(b)))
    for i in range(len(a)):
        for j in range(len(b)):
            out[i, j] = abs(a[i] - b[j])
    return out

def broadcast():
    return np.abs(a[:, None] - b[None, :])

print(np.allclose(nested(), broadcast()))
print(f'nested: {timeit(nested, 1)*1000:.0f} ms, broadcast: {timeit(broadcast)*1000:.2f} ms')

True
nested: 41 ms, broadcast: 0.25 ms


### Grouped sums → `bincount` with weights

In [12]:
group = np.array([0, 2, 1, 0, 2, 2])
amount = np.array([10., 5., 7., 3., 1., 4.])
np.bincount(group, weights=amount)    # total amount per group

array([13.,  7., 10.])

## What about `np.vectorize`?

`np.vectorize` wraps a Python function so it accepts arrays. It's convenient, but it's still a Python loop under the hood, so **it doesn't make anything faster**. Use it for convenience, not performance.

In [13]:
def grade(score):
    if score >= 90: return 'A'
    if score >= 80: return 'B'
    if score >= 70: return 'C'
    return 'F'

scores = rng.integers(50, 100, size=8)
print(scores)
print(np.vectorize(grade)(scores))

[58 88 51 61 53 87 61 81]
['F' 'B' 'F' 'F' 'F' 'B' 'F' 'B']


The truly vectorized version of the same thing:

In [14]:
labels = np.array(['F', 'C', 'B', 'A'])
labels[np.searchsorted([70, 80, 90], scores, side='right')]

array(['F', 'B', 'F', 'F', 'F', 'B', 'F', 'B'], dtype='<U1')

## When you really can't vectorize

Some algorithms depend on the previous step's result in a way that doesn't map to `cumsum`/`accumulate`. In those cases, look at [Numba](https://numba.pydata.org/), which compiles a plain Python loop to machine code, or write that part in Cython.

# 14. Missing Values: NaN and inf

Real data has gaps. NumPy doesn't have a dedicated "missing" value for numbers, so the convention is to use `np.nan` (not a number). This notebook covers how NaN behaves and how to work with it.

In [1]:
import numpy as np

data = np.array([2.0, np.nan, 5.0, 1.0, np.nan, 7.0])
data

array([ 2., nan,  5.,  1., nan,  7.])

## NaN spreads

Any arithmetic involving NaN gives NaN, so one missing value ruins a normal sum or mean:

In [2]:
print(data.sum(), data.mean(), data.max())

nan nan nan


## NaN isn't equal to anything, including itself

So `data == np.nan` never finds anything. Use `np.isnan`:

In [3]:
print(np.nan == np.nan)
print(data == np.nan)
print(np.isnan(data))

False
[False False False False False False]
[False  True False False  True False]


In [4]:
print('missing:', np.isnan(data).sum())
print('positions:', np.flatnonzero(np.isnan(data)))

missing: 2
positions: [1 4]


NaN only exists for floats. An integer array can't hold it, which is why integer data with gaps usually ends up as floats:

In [5]:
try:
    np.array([1, 2, 3])[0] = np.nan
except ValueError as e:
    print('ValueError:', e)

ValueError: cannot convert float NaN to integer


## NaN-aware functions

Most reductions have a `nan*` version that skips missing values:

In [6]:
print(np.nansum(data))
print(np.nanmean(data))
print(np.nanmax(data), np.nanmin(data))
print(np.nanstd(data))
print(np.nanmedian(data))
print(np.nanpercentile(data, 50))
print(np.nanargmax(data))

15.0
3.75
7.0 1.0
2.384848003542364
3.5
3.5
5


They work along axes too:

In [7]:
table = np.array([[1.0, np.nan, 3.0],
                  [4.0, 5.0, 6.0],
                  [np.nan, 8.0, 9.0]])
print(np.nanmean(table, axis=0))

[2.5 6.5 6. ]


## Dropping missing values

In [8]:
print(data[~np.isnan(data)])

[2. 5. 1. 7.]


In [9]:
# drop rows that contain any NaN
rows_ok = ~np.isnan(table).any(axis=1)
table[rows_ok]

array([[4., 5., 6.]])

## Filling missing values

In [10]:
print(np.nan_to_num(data, nan=0.0))
print(np.where(np.isnan(data), np.nanmean(data), data))

[2. 0. 5. 1. 0. 7.]
[2.   3.75 5.   1.   3.75 7.  ]


Filling each column with its own mean, using broadcasting:

In [11]:
col_means = np.nanmean(table, axis=0)
filled = np.where(np.isnan(table), col_means, table)
filled

array([[1. , 6.5, 3. ],
       [4. , 5. , 6. ],
       [2.5, 8. , 9. ]])

Linear interpolation for gaps in a 1-D series:

In [12]:
series = np.array([1.0, np.nan, np.nan, 4.0, 5.0, np.nan, 7.0])
idx = np.arange(len(series))
known = ~np.isnan(series)
np.interp(idx, idx[known], series[known])

array([1., 2., 3., 4., 5., 6., 7.])

## Infinity

Dividing by zero gives `inf` (or `nan` for 0/0). NumPy warns instead of raising. `np.isinf` and `np.isfinite` let you check for them:

In [13]:
with np.errstate(divide='ignore', invalid='ignore'):
    x = np.array([1.0, 0.0, -1.0]) / 0.0
print(x)
print(np.isinf(x), np.isnan(x), np.isfinite(x))

[ inf  nan -inf]
[ True False  True] [False  True False] [False False False]


`np.errstate` controls what happens on these floating point problems. You can silence the warnings (as above) or turn them into errors, which is great for tracking down where a NaN first appears:

In [14]:
try:
    with np.errstate(all='raise'):
        np.log(np.array([0.0]))
except FloatingPointError as e:
    print('FloatingPointError:', e)

FloatingPointError: divide by zero encountered in log


## Masked arrays

`numpy.ma` is the other approach: keep the data as is and attach a mask saying which values to ignore. It works with any dtype, including integers, and most functions respect the mask automatically.

In [15]:
readings = np.array([12, -999, 15, 14, -999, 13])    # -999 = sensor error
m = np.ma.masked_equal(readings, -999)
print(m)
print(m.mean())
print(m.filled(0))

[12 -- 15 14 -- 13]
13.5
[12  0 15 14  0 13]


Honestly, for most tabular data with lots of missing values, pandas is more comfortable. But for numerical arrays, NaN plus the `nan*` functions covers almost everything.

# 15. Saving and Loading Arrays

NumPy has its own binary format (`.npy` / `.npz`) and can also read and write plain text like CSV.

This notebook writes everything into a temporary folder so it doesn't leave files lying around.

In [1]:
import numpy as np
import tempfile, os

tmp = tempfile.mkdtemp()
path = lambda name: os.path.join(tmp, name)

rng = np.random.default_rng(0)
arr = rng.random((3, 4))

## `.npy`: a single array

Binary, fast, and keeps the exact dtype and shape. This is what you should use for saving arrays between your own Python sessions.

In [2]:
np.save(path('arr.npy'), arr)
loaded = np.load(path('arr.npy'))
print(np.array_equal(arr, loaded), loaded.dtype, loaded.shape)

True float64 (3, 4)


## `.npz`: several arrays in one file

`savez` stores multiple arrays under names. `savez_compressed` does the same with zip compression, which helps a lot for data with repetition.

In [3]:
labels = np.array([0, 1, 1])
np.savez(path('dataset.npz'), features=arr, labels=labels)

with np.load(path('dataset.npz')) as f:
    print(f.files)
    print(f['labels'])

['features', 'labels']
[0 1 1]


In [4]:
sparse_ish = np.zeros((1000, 1000))
np.savez(path('plain.npz'), a=sparse_ish)
np.savez_compressed(path('small.npz'), a=sparse_ish)
for name in ['plain.npz', 'small.npz']:
    print(name, os.path.getsize(path(name)) // 1024, 'KB')

plain.npz 7812 KB
small.npz 7 KB


## Memory mapping big files

For arrays too big to fit in RAM, `mmap_mode` lets you open a `.npy` file and only read the parts you touch:

In [5]:
np.save(path('big.npy'), np.arange(10_000_000, dtype=np.int32))
mm = np.load(path('big.npy'), mmap_mode='r')
print(type(mm).__name__, mm[5_000_000:5_000_005])

memmap [5000000 5000001 5000002 5000003 5000004]


## Text files

Text is slower and bigger, but human-readable and works with anything (Excel, other languages, ...).

### Writing with `savetxt`

In [6]:
np.savetxt(path('arr.csv'), arr, delimiter=',', fmt='%.4f',
           header='a,b,c,d', comments='')
print(open(path('arr.csv')).read())

a,b,c,d
0.6370,0.2698,0.0410,0.0165
0.8133,0.9128,0.6066,0.7295
0.5436,0.9351,0.8159,0.0027



### Reading with `loadtxt`

`loadtxt` is fine for clean, purely numeric files:

In [7]:
back = np.loadtxt(path('arr.csv'), delimiter=',', skiprows=1)
back

array([[0.637 , 0.2698, 0.041 , 0.0165],
       [0.8133, 0.9128, 0.6066, 0.7295],
       [0.5436, 0.9351, 0.8159, 0.0027]])

In [8]:
# pick specific columns
np.loadtxt(path('arr.csv'), delimiter=',', skiprows=1, usecols=(0, 2))

array([[0.637 , 0.041 ],
       [0.8133, 0.6066],
       [0.5436, 0.8159]])

### Messier files with `genfromtxt`

`genfromtxt` handles missing values and can use the header row as field names:

In [9]:
csv = '''name,age,score
alice,30,88.5
bob,,92.0
carol,25,
'''
with open(path('people.csv'), 'w') as f:
    f.write(csv)

people = np.genfromtxt(path('people.csv'), delimiter=',', names=True,
                       dtype=None, encoding='utf-8')
print(people)
print(people['age'])
print(people['score'])

[('alice', 30, 88.5) ('bob', -1, 92. ) ('carol', 25,  nan)]
[30 -1 25]
[88.5 92.   nan]


Missing numbers come back as `-1` for ints and `nan` for floats by default (you can change this with `filling_values`). The result is a **structured array**, which you index by column name. For anything more complicated than this, use pandas (`pd.read_csv`) instead.

## Which format should I use?

| Situation | Use |
|---|---|
| Saving work to reload in Python later | `np.save` / `np.load` |
| Several related arrays | `np.savez` (or `savez_compressed`) |
| Data bigger than memory | `.npy` + `mmap_mode` |
| Sharing with people or other tools | `savetxt` / CSV |
| Mixed column types, messy CSV | pandas |

One more thing: `np.load` won't load files containing Python objects unless you pass `allow_pickle=True`. Only do that for files you trust, since unpickling can run arbitrary code.

In [10]:
import shutil
shutil.rmtree(tmp)